# Step 1 - Prepare MedEV Test Data

Load MedEV from Hugging Face and create an evaluation file for Vietnamese -> English MT.

Output:
- `data/medev_test.csv` with columns: `id | source_vi | reference_en`

In [3]:
import os
import pandas as pd
from datasets import load_dataset

DATASET_NAME = "nhuvo/MedEV"
OUTPUT_PATH = os.path.join("data", "medev_test.csv")
os.makedirs("data", exist_ok=True)

MAX_EVAL_SAMPLES = None  # set e.g. 500 for quick run
RANDOM_SEED = 42

print(f"Dataset: {DATASET_NAME}")

Dataset: nhuvo/MedEV


In [4]:
import re

ds = load_dataset(DATASET_NAME)
print(ds)

sample = ds["test"][0]
print("Test columns:", list(sample.keys()))

if "text" in sample and len(sample.keys()) == 1:
    # MedEV text-only format on HF: first half English, second half Vietnamese
    raw = ds["test"].to_pandas()["text"].dropna().astype(str).reset_index(drop=True)
    n = len(raw)
    half = n // 2

    if n % 2 != 0:
        raise ValueError(f"Unexpected odd number of rows in text-only split: {n}")

    en_series = raw.iloc[:half].reset_index(drop=True)
    vi_series = raw.iloc[half:].reset_index(drop=True)

    # Light sanity check for Vietnamese characters in second half
    vi_pattern = re.compile(r"[ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩịóòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ]", re.IGNORECASE)
    vi_hits = vi_series.head(min(200, len(vi_series))).apply(lambda x: bool(vi_pattern.search(x))).sum()
    if vi_hits == 0:
        raise ValueError("Text-only format detected but Vietnamese half sanity check failed.")

    medev = pd.DataFrame({
        "source_vi": vi_series,
        "reference_en": en_series,
    })
else:
    candidate_vi = ["vi", "src", "source", "sentence_vi", "input"]
    candidate_en = ["en", "tgt", "target", "sentence_en", "output", "reference"]

    vi_col = next((c for c in candidate_vi if c in sample), None)
    en_col = next((c for c in candidate_en if c in sample), None)

    if vi_col is None or en_col is None:
        raise ValueError("Could not auto-detect Vietnamese/English columns.")

    print(f"Detected columns -> Vietnamese: {vi_col} | English: {en_col}")
    medev = ds["test"].to_pandas()[[vi_col, en_col]].rename(
        columns={vi_col: "source_vi", en_col: "reference_en"}
    )

medev = medev.dropna().drop_duplicates().reset_index(drop=True)

if MAX_EVAL_SAMPLES is not None and MAX_EVAL_SAMPLES < len(medev):
    medev = medev.sample(MAX_EVAL_SAMPLES, random_state=RANDOM_SEED).reset_index(drop=True)

medev.insert(0, "id", range(1, len(medev) + 1))

print(f"Rows prepared: {len(medev):,}")
medev.head(8)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 681794
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 17878
    })
    test: Dataset({
        features: ['text'],
        num_rows: 17920
    })
})
Test columns: ['text']
Rows prepared: 8,959


,id,source_vi,reference_en
0,1,Thực trạng kiến thức và thực hành của người có...,"Knowledge, practices in public health service ..."
1,2,"Mô tả thực trạng kiến thức, thực hành của ngườ...","Describe knowledge, practices in public health..."
2,3,Phương pháp: Thiết kế nghiên mô tả cắt ngang đ...,Methodology: A cross sectional study was used ...
3,4,Kết quả: Tỷ lệ người biết được khám chữa bệnh ...,Results: Percentage of card's holders who knew...
4,5,Tỷ lệ người có thẻ BHYT thực hành khám chữa bệ...,Percentage of card's holders who went to the f...
5,6,Tỷ lệ người có thẻ BHYT sử dụng thẻ để lấy thu...,Percentage of card's holders who went to publi...
6,7,"Các yếu tố khoảng cách từ nhà đến cơ sở y tế, ...",The determinants of knowledge and practices in...
7,8,Kết luận: Kiến thức và thực hành của người có ...,Conclusions: Knowledge and practices in public...


In [5]:
medev.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
print(f"Saved -> {OUTPUT_PATH}")
print(medev.shape)
medev.tail(5)

Saved -> data\medev_test.csv
(8959, 3)


,id,source_vi,reference_en
8954,8955,Các tác giả xây dựng các giải pháp và tiến hàn...,The pilot intervention was implemented at 4 pr...
8955,8956,Xây dựng các thực đơn cho trẻ TC - BP theo từn...,To setting up the menu for the school children...
8956,8957,Hướng dẫn các phương pháp luyện tập thể dục th...,To guideline for the physical activity at the ...
8957,8958,Có sự thay đổi về các tập quán ăn uống và thói...,There was a changes of the food habits and hab...
8958,8959,"Các thói quen ăn nhanh, ăn nhiều, ăn thêm bữa ...","The habit of fast eating, eat planty of food, ..."
